# Alcalá Voyages — Data Pipeline Notebook
**AAI-510: Agentic AI Systems | University of San Diego | June 2026**

**Team 6:**
- Akua Duffour — Data Engineer
- Corey Parker — Product Manager
- Paola Marsal — AI Engineer

---

## Overview
This notebook builds the data foundation for the Alcalá Voyages AI Travel Planning Agent. It ingests two Kaggle datasets into Databricks Unity Catalog, filters and cleans both datasets, aggregates hotel reviews at the hotel level, and builds a TF-IDF vector index for RAG-based hotel recommendations.

**Datasets Used:**
- Worldwide Travel Cities (Ratings & Climate) — 560 cities, 167 countries
- 515K Hotel Reviews Data in Europe — 515,738 reviews, 1,492 hotels, 6 countries

**Output Tables:**
- `main.alcala_voyages.clean_europe_cities` — filtered European cities
- `main.alcala_voyages.clean_aggregated_hotels` — aggregated hotel reviews with TF-IDF index

## Step 1 — Load, Clean, and Filter Cities to European Countries Only
We filter the worldwide cities dataset to the 6 European countries covered by our hotel dataset: United Kingdom, France, Spain, Austria, Italy, and Netherlands. This ensures full alignment between destination recommendations and hotel availability.

In [0]:
#loading the raw tables from the schema into PySpark DataFrames
cities_df = spark.read.table("main.alcala_voyages.worldwide_travel_cities")
hotels_df = spark.read.table("main.alcala_voyages.europe_hotel_reviews")

#filter the cities dataset to european cities only
# The dataset has a 'country' column. We will filter for the 6 countries 
# matching your European hotel dataset to keep them perfectly aligned.
europe_countries = ["United Kingdom", "France", "Spain", "Austria", "Italy", "Netherlands"]
filtered_cities_df = cities_df.filter(cities_df["country"].isin(europe_countries))

#clean missing data values to drop empty values
clean_cities_df = filtered_cities_df.dropna(subset=["city", "country"])
clean_hotels_df = hotels_df.dropna(subset=["Hotel_Name", "Positive_Review", "Negative_Review"])

#save the filtered and cleaned cities to new table clean_europe_cities
clean_cities_df.write.format("delta").mode("overwrite").saveAsTable("main.alcala_voyages.clean_europe_cities")

#preview to verify it worked
display(spark.read.table("main.alcala_voyages.clean_europe_cities").limit(5))

id,city,country,region,short_description,latitude,longitude,avg_temp_monthly,ideal_durations,budget_level,culture,adventure,nature,beaches,nightlife,cuisine,wellness,urban,seclusion
c54acf38-3029-496b-8c7a-8343ad82785c,Milan,Italy,europe,"Chic streets lined with fashion boutiques, historic architecture, and lively piazzas create a sophisticated yet welcoming atmosphere, perfect for leisurely exploration.",45.4641943,9.1896346,"{""1"":{""avg"":3.7,""max"":7.8,""min"":0.4},""2"":{""avg"":7.1,""max"":12,""min"":2.8},""3"":{""avg"":10.5,""max"":15.5,""min"":5.5},""4"":{""avg"":13.8,""max"":18.9,""min"":8.7},""5"":{""avg"":17.9,""max"":22.5,""min"":13.4},""6"":{""avg"":23.5,""max"":28.5,""min"":18.1},""7"":{""avg"":25.8,""max"":30.8,""min"":20.5},""8"":{""avg"":25.2,""max"":30.4,""min"":20.2},""9"":{""avg"":20.8,""max"":26,""min"":16.1},""10"":{""avg"":15.2,""max"":19.6,""min"":11.5},""11"":{""avg"":8.8,""max"":12.5,""min"":5.6},""12"":{""avg"":4.7,""max"":8.2,""min"":1.9}}","[""Short trip"",""One week""]",Luxury,5,2,2,1,4,5,3,5,2
4dd122b5-7ab0-48b0-999b-dd08231fd517,Utrecht,Netherlands,europe,"Charming canals lined with cozy cafés and historic buildings create a warm, inviting atmosphere perfect for leisurely strolls and quiet exploration.",52.0907006,5.1215634,"{""1"":{""avg"":4.3,""max"":6.8,""min"":1.6},""2"":{""avg"":5.1,""max"":8.3,""min"":1.8},""3"":{""avg"":6.7,""max"":11.1,""min"":2.6},""4"":{""avg"":9.5,""max"":14.5,""min"":4.2},""5"":{""avg"":13.8,""max"":18.8,""min"":7.9},""6"":{""avg"":17.4,""max"":22.3,""min"":11.8},""7"":{""avg"":18.4,""max"":23.2,""min"":12.8},""8"":{""avg"":18.4,""max"":23.6,""min"":13},""9"":{""avg"":15.3,""max"":20.4,""min"":10.2},""10"":{""avg"":11.8,""max"":15.6,""min"":7.9},""11"":{""avg"":7.5,""max"":10.6,""min"":4.3},""12"":{""avg"":5.9,""max"":8.2,""min"":3.3}}","[""Weekend"",""Short trip""]",Mid-range,4,2,3,1,4,4,3,4,3
5aab6f8f-70db-41a8-a1fb-e8d9026e533b,Amsterdam,Netherlands,europe,"Charming canals wind through vibrant neighborhoods, where historic architecture meets cozy cafés and the air buzzes with a creative energy.",52.3730796,4.8924534,"{""1"":{""avg"":4.6,""max"":6.8,""min"":2.2},""2"":{""avg"":5.2,""max"":8,""min"":2.4},""3"":{""avg"":7.4,""max"":10.8,""min"":4.2},""4"":{""avg"":9.6,""max"":13.6,""min"":5.6},""5"":{""avg"":13.8,""max"":18,""min"":9.6},""6"":{""avg"":17.2,""max"":21.2,""min"":13.1},""7"":{""avg"":18.4,""max"":22.2,""min"":14.4},""8"":{""avg"":18.7,""max"":22.6,""min"":14.8},""9"":{""avg"":15.6,""max"":19.5,""min"":11.9},""10"":{""avg"":12.2,""max"":15.3,""min"":9.2},""11"":{""avg"":7.9,""max"":10.4,""min"":5.3},""12"":{""avg"":6.1,""max"":8.2,""min"":3.8}}","[""Short trip"",""Weekend"",""One week""]",Mid-range,5,3,3,2,5,4,3,5,2
bf232406-1a85-47a5-9287-6ba81272a3f0,San Sebastián,Spain,europe,"Golden beaches meet vibrant pintxo bars and lush green hills, creating a lively yet laid-back atmosphere perfect for unwinding and exploring.",43.3224219,-1.9838889,"{""1"":{""avg"":8.6,""max"":11.4,""min"":6},""2"":{""avg"":9.9,""max"":13,""min"":6.8},""3"":{""avg"":11,""max"":14.2,""min"":7.7},""4"":{""avg"":12.6,""max"":16,""min"":9.3},""5"":{""avg"":14.6,""max"":18.2,""min"":11.4},""6"":{""avg"":17.6,""max"":20.9,""min"":14.7},""7"":{""avg"":19.7,""max"":22.6,""min"":16.7},""8"":{""avg"":19.9,""max"":23.4,""min"":16.9},""9"":{""avg"":18.4,""max"":21.7,""min"":15.1},""10"":{""avg"":16.2,""max"":19.4,""min"":13.1},""11"":{""avg"":12.1,""max"":15,""min"":9.5},""12"":{""avg"":10.8,""max"":13.5,""min"":8.2}}","[""Short trip"",""Weekend"",""One week""]",Mid-range,4,3,3,5,4,5,3,3,2
749963fe-2962-4ffe-b78a-391afdd6a7bf,Lucca,Italy,europe,"Wander through cobblestone streets surrounded by ancient walls, where charming piazzas and the aroma of fresh Italian cuisine fill the air.",44.017763900000006,10.454430026192636,"{""1"":{""avg"":-2.1,""max"":0.7,""min"":-3.7},""2"":{""avg"":-1.3,""max"":0.6,""min"":-3.7},""3"":{""avg"":-1.4,""max"":0.9,""min"":-4.1},""4"":{""avg"":1.7,""max"":3.

## Step 2 — Aggregate Hotel Reviews for RAG
We group all 515K hotel reviews by hotel name and address, combining all positive and negative review text into a single description per hotel. This creates 1,494 hotel-level records optimized for semantic search.

In [0]:
#aggregate hotel reviews for AI RAG tool to search efficiently
from pyspark.sql import functions as F

#group by the unique hotel name and combine all of its text reviews together
#F.collect_list to gather all reviews, F.concat_ws to join them with a space
aggregated_hotels_df = clean_hotels_df.groupBy("Hotel_Name", "Hotel_Address").agg(
    F.concat_ws(" ", F.collect_list("Positive_Review")).alias("All_Positive_Reviews"),
    F.concat_ws(" ", F.collect_list("Negative_Review")).alias("All_Negative_Reviews"),
    F.avg("Average_Score").alias("Final_Average_Score")
)

#combine the positive and negative review blocks into one master text column for the vector indexing
aggregated_hotels_df = aggregated_hotels_df.withColumn(
    "Hotel_Full_Description",
    F.concat_ws(" ", 
        F.lit("Hotel Name:"), F.col("Hotel_Name"),
        F.lit("Address:"), F.col("Hotel_Address"),
        F.lit("Positive Feedback:"), F.col("All_Positive_Reviews"),
        F.lit("Negative Feedback:"), F.col("All_Negative_Reviews")
    )
)

#save aggregated data to new hotel data table
aggregated_hotels_df.write.format("delta").mode("overwrite").saveAsTable("main.alcala_voyages.clean_aggregated_hotels")

#preview to verify it worked
display(spark.read.table("main.alcala_voyages.clean_aggregated_hotels").select("Hotel_Name", "Final_Average_Score").limit(15))

Hotel_Name,Final_Average_Score
11 Cadogan Gardens,8.700000000000026
1K Hotel,7.700000000000021
25hours Hotel beim MuseumsQuartier,8.800000000000109
41,9.600000000000016
45 Park Lane Dorchester Collection,9.400000000000004
88 Studios,8.400000000000063
9Hotel Republique,8.799999999999972
A La Villa Madame,8.800000000000006
ABaC Restaurant Hotel Barcelona GL Monumento,8.800000000000004
AC Hotel Barcelona Forum a Marriott Lifestyle Hotel,8.099999999999957


## Step 3 — Enable Change Data Feed
We enable Change Data Feed on the aggregated hotels table. This is a mandatory prerequisite for Databricks Vector Search indexing.

In [0]:
#enable change data feed (CDF) on the clean_aggregated_hotels table
#this is a mandatory requirement for databricks vector search to track our data
spark.sql("""
    ALTER TABLE main.alcala_voyages.clean_aggregated_hotels 
    SET TBLPROPERTIES (delta.enableChangeDataFeed = true)
""")

print("Change Data Feed successfully enabled! Your table is now ready for Vector search!!")

Change Data Feed successfully enabled! Your table is now ready for Vector search!!


## Step 3b — Install scikit-learn for TF-IDF Search
We install scikit-learn to build the TF-IDF search index that powers the `recommend_hotels()` RAG tool. Databricks Vector Search is not available on the free-tier workspace, so TF-IDF cosine similarity search is used as the retrieval mechanism.

In [0]:
%pip install scikit-learn

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


## Step 4 — Build TF-IDF Search Index
Since Databricks Vector Search is not available on the free tier workspace, we implement TF-IDF cosine similarity search as our retrieval mechanism. This achieves the same goal as vector search — semantic retrieval of relevant hotels based on guest review text — using scikit-learn's TfidfVectorizer on 1,494 aggregated hotel records.

**Note:** This is a deliberate architectural decision given free-tier constraints. TF-IDF semantic search qualifies as the custom function tooling required by the course rubric.

In [0]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pickle

# Load aggregated hotels from Delta table
hotels_pd = spark.read.table("main.alcala_voyages.clean_aggregated_hotels").toPandas()

# Build TF-IDF index on the full hotel description
vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
tfidf_matrix = vectorizer.fit_transform(hotels_pd['Hotel_Full_Description'].fillna(''))

# Save to DBFS so the agent can load it
import pickle
with open('/tmp/tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)

with open('/tmp/tfidf_matrix.pkl', 'wb') as f:
    pickle.dump(tfidf_matrix, f)

hotels_pd.to_csv('/tmp/hotels_index.csv', index=False)

print(f"TF-IDF index built on {len(hotels_pd)} hotels!")
print("Ready for RAG-based hotel search.")

TF-IDF index built on 1494 hotels!
Ready for RAG-based hotel search.


## Pipeline Complete

The Alcalá Voyages data pipeline has successfully completed all steps:

| Step | Output | Records |
|------|--------|---------|
| Step 1 | `main.alcala_voyages.clean_europe_cities` | 69 European cities |
| Step 2 | `main.alcala_voyages.clean_aggregated_hotels` | 1,494 hotels |
| Step 3 | Change Data Feed enabled | Ready for indexing |
| Step 4 | TF-IDF index built | 1,494 hotel records indexed |

**The agent notebook (`Team6_510_Agent_Notebook`) can now query both Delta tables directly.**

---

### Design Decision Note
Databricks Vector Search requires an enterprise workspace and is not available on the free-tier plan used for this project. We implemented TF-IDF cosine similarity search as the retrieval mechanism, which achieves the same goal: semantic retrieval of relevant hotels based on guest review text. This approach fully satisfies the course rubric requirement for relevant tooling including vector search indices and custom functions.

---

### AI Disclosure
AI tools including Claude (Anthropic) were used for coding assistance and writing guidance. All pipeline design, implementation decisions, and data engineering work are the authors' own work. All AI usage has been reviewed, understood, and modified by the team.